<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Classification metrics

**[C05-M03-L03 · Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/)** · C05, Machine Learning: From Problem to Reliable Model · Module 3, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** count the four outcomes of a yes-or-no classifier in a confusion matrix, compute accuracy, precision, recall and F1, see why accuracy misleads with a rare class, and choose the metric from the decision.

| | |
|---|---|
| **Time** | About 60 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Regression errors, from [Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/). The baselines and the 20% capacity, from [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). The validation set, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/). |
| **You do not need** | The local project. The setup below downloads the data. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M03-L03-classification-metrics/classification-metrics-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Count, do not subtract

A regression error is real − predicted. Larkfield's escalation question has a yes-or-no answer, and "escalated" minus "not escalated" has no meaning. So we **count** what happened. Two answers and two truths give four outcomes, as for a smoke alarm (the [lesson page](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/) has the picture):

| What happened | Name | Short |
|---|---|---|
| Flagged, and it escalated (caught) | true positive | TP |
| Not flagged, but it escalated (missed) | false negative | FN |
| Flagged, but it did not escalate (an unnecessary review) | false positive | FP |
| Not flagged, and it did not escalate | true negative | TN |

## 2. A points score and ten tickets

No model exists yet. Priya builds a **points score**, a checklist like a hospital's early-warning score. Each fact that makes an escalation more likely adds points:

- priority 1: **+2 points**;
- team is payment or warranty: **+1**;
- at least one escalation of the customer in the last 90 days: **+1**.

The next cell defines the score as a function, and applies it to ten made-up tickets. A ticket is **flagged** when it has **3 points or more**.

> **Predict.** T-3 has priority 1, team warranty and no earlier escalation. How many points does it get? Is it flagged? Then run the cell.

In [ ]:
TARGET = "escalated_72h"

def points_score(tickets):
    """Larkfield's points score: 0 to 4 points per ticket."""
    points = 2 * (tickets["priority"] == 1)
    points += tickets["team"].isin(["payment", "warranty"])
    points += tickets["prior_escalations_90d"] >= 1
    return points.astype(int).rename("points")

tickets_10 = pd.DataFrame({
    "ticket": [f"T-{i}" for i in range(1, 11)],
    "priority": [1, 1, 1, 1, 3, 2, 3, 2, 3, 3],
    "team": ["payment", "delivery", "warranty", "returns", "payment", "account", "delivery", "warranty", "returns", "account"],
    "prior_escalations_90d": [1, 0, 0, 1, 0, 0, 2, 1, 0, 1],
    "escalated_72h": [1, 0, 1, 0, 1, 0, 0, 1, 0, 0],
})
tickets_10["points"] = points_score(tickets_10)
tickets_10["flagged"] = (tickets_10["points"] >= 3).astype(int)
print(tickets_10.to_string(index=False))

> **Check.** You should see the points 4, 2, 3, 3, 1, 0, 1, 2, 0, 1. T-3 has 2 + 1 + 0 = 3 points, so it is flagged. Three tickets are flagged: T-1, T-3 and T-4.

Now count the four outcomes. Run the cell.

In [ ]:
y_10 = tickets_10[TARGET] == 1
flags_10 = tickets_10["flagged"] == 1
caught, false_alarms = int((flags_10 & y_10).sum()), int((flags_10 & ~y_10).sum())
missed, correctly_left = int((~flags_10 & y_10).sum()), int((~flags_10 & ~y_10).sum())
print(f"caught {caught}, false alarms {false_alarms}, missed {missed}, correctly left {correctly_left}")
print("precision", round(caught / (caught + false_alarms), 3), "| recall", caught / (caught + missed),
      "| accuracy", (caught + correctly_left) / 10)
print("never flag: accuracy", correctly_left + false_alarms, "/ 10, recall 0")

> **Check.** You should see `caught 2, false alarms 1, missed 2, correctly left 5`, then `precision 0.667 | recall 0.5 | accuracy 0.7`, then `never flag: accuracy 6 / 10, recall 0`.

The technical names: caught = **true positive (TP)**, false alarm = **false positive (FP)**, missed = **false negative (FN)**, correctly left = **true negative (TN)**.

- **Accuracy** = (TP + TN) ÷ all: the share of right answers.
- **Precision** = TP ÷ (TP + FP): when it flags, how often is it right? False alarms lower it.
- **Recall** = TP ÷ (TP + FN): of the real escalations, how many did it catch? Misses lower it.

A rule that never flags gets 0.6 accuracy here, close to the points score's 0.7, and it catches nothing. With a rare positive class, accuracy rewards saying "no".

**F1** combines precision and recall: 2 × precision × recall ÷ (precision + recall). It is high only when **both** are high.

> **Predict.** A classifier has precision 0.9 and recall 0.1. The plain mean of the two is 0.5. Is F1 higher or lower than 0.5? Then run the cell.

In [ ]:
for precision, recall in [(0.9, 0.1), (0.5, 0.5), (2 / 3, 0.5)]:
    print(round(precision, 3), round(recall, 3), "-> F1", round(2 * precision * recall / (precision + recall), 3))

> **Check.** You should see `-> F1 0.18`, `-> F1 0.5` and `-> F1 0.571`. The weaker of the two numbers pulls F1 down.

## 3. The worked example: why these points, and how they do on May

Priya chose the points from the **train set** only: each fact must make an escalation clearly more likely. The next cell loads the data, with `prior_escalations_90d` from `extra_features.csv`, and prints the escalation rate on train with and without each fact.

> **Predict.** On train, 11.7% of tickets escalate. Which fact do you expect to raise the rate the most? Then run the cell.

In [ ]:
def load(part):
    """One part of the split, with prior_escalations_90d from extra_features.csv."""
    tickets = pd.read_csv(f"data/{part}.csv", keep_default_na=False, na_values=[""])
    extra = pd.read_csv("data/extra_features.csv", usecols=["ticket_id", "prior_escalations_90d"])
    return tickets.merge(extra, on="ticket_id", how="left")

train, valid = load("train"), load("valid")
facts = {"priority 1": train["priority"] == 1,
         "payment or warranty": train["team"].isin(["payment", "warranty"]),
         "an escalation in 90 days": train["prior_escalations_90d"] >= 1,
         "(not used) 150 words or more": train["word_count"] >= 150,
         "(not used) phone": train["channel"] == "phone"}
for name, has in facts.items():
    print(f"{name:30s} {has.sum():5d} tickets | with: {train[TARGET][has].mean():.1%} | without: {train[TARGET][~has].mean():.1%}")

> **Check.** You should see priority 1 with 2,385 tickets, 29.1% with and 8.9% without. The other two facts give 18.5% and 14.8% with the fact. Long messages (18.8%) and the phone channel (16.5%) help too, but Priya keeps the checklist to three quick questions.

Priority 1 is the strongest fact, so it gets 2 points; each of the other two gets 1 point. Now the score on May, the validation set. Run the cell.

In [ ]:
print(valid.groupby(points_score(valid))[TARGET].agg(tickets="size", rate="mean").round(3))

> **Check.** You should see five rows, 0 to 4 points, with 320, 532, 211, 84 and 32 tickets. The rate rises from 0.053 (0 points) to 0.562 (4 points): the score sorts the May tickets as it sorted the train tickets.

The next cell defines `flag_report()`, the function of [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/) with the same plain words, and F1 added. Then it compares three classifiers on May.

> **Predict.** Which of the three has the highest accuracy: never escalate, flag priority 1, or 3 points or more? Then run the cell.

In [ ]:
def flag_report(y, flags):
    """Compare 0/1 flags with the real 0/1 answers: four counts and five measures."""
    y, flags = np.asarray(y, dtype=bool), np.asarray(flags, dtype=bool)
    caught, false_alarms = int((flags & y).sum()), int((flags & ~y).sum())
    missed, correctly_left = int((~flags & y).sum()), int((~flags & ~y).sum())
    precision = caught / (caught + false_alarms) if caught + false_alarms else None
    recall = caught / (caught + missed)
    f1 = 2 * precision * recall / (precision + recall) if caught else 0.0
    return {"flagged": caught + false_alarms, "share flagged": round((caught + false_alarms) / len(y), 3),
            "caught": caught, "false alarms": false_alarms, "missed": missed, "correctly left": correctly_left,
            "precision": round(precision, 3) if precision is not None else None, "recall": round(recall, 3),
            "F1": round(f1, 3), "accuracy": round((caught + correctly_left) / len(y), 3)}

y = valid[TARGET]
points = points_score(valid)
reports = pd.DataFrame([flag_report(y, np.zeros(len(valid))),
                        flag_report(y, valid["priority"] == 1),
                        flag_report(y, points >= 3)],
                       index=["never escalate", "rule: priority 1", "points: 3 or more"])
print(reports.to_string())

> **Check.** You should see three rows:
>
> - never escalate: flagged 0, recall 0.000, accuracy 0.845;
> - rule: priority 1: flagged 163, precision 0.374, recall 0.333, F1 0.353, accuracy 0.810;
> - points: 3 or more: flagged 116, caught 53, precision 0.457, recall 0.290, F1 0.355, accuracy 0.836.

"Never escalate" has the highest accuracy and catches nothing. The points score has the better precision, the rule the better recall, but they flag different numbers of tickets (116 against 163). The next lesson compares them at the same workload.

## 4. The confusion matrix with scikit-learn

scikit-learn's `confusion_matrix(y, flags)` counts the four outcomes for you. It puts the real class in the rows and the answer in the columns, **class 0 first**: `[[TN, FP], [FN, TP]]`. Run the cell.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

print(confusion_matrix(y, points >= 3))

> **Check.** You should see `[[933  63]` and ` [130  53]]`: TN 933, FP 63, FN 130 and TP 53.

> **Your turn.** Unpack the four numbers with `tn, fp, fn, tp = confusion_matrix(y, points >= 3).ravel()`. Compute `precision_3` and `recall_3` from them, each rounded to 3 decimals. Then run the check cell.

In [ ]:
precision_3 = ...  # TP / (TP + FP), rounded to 3 decimals
recall_3 = ...     # TP / (TP + FN), rounded to 3 decimals

In [ ]:
expect_hash('(precision_3, recall_3)', (precision_3, recall_3), '11fc7c3bb0517c12')

`precision_score(y, points >= 3)` and `recall_score(y, points >= 3)` give the same numbers. Always read the labels of a confusion matrix before you read the numbers: this lesson's tables start with the escalated row, scikit-learn's matrix with class 0.

## 5. Change one thing: a wider cut-off

Flag the tickets with **2 points or more**.

> **Predict.** Will precision go up or down? Recall? F1? Will the flags fit the capacity of 20%?

> **Your turn.** Put `flag_report(y, points >= 2)` in `report_2`, print it, and run the check cell.

In [ ]:
report_2 = ...  # flag_report() for 2 points or more

In [ ]:
expect_hash('report_2', report_2, '11d5405382857d21')

A wider cut-off flags more tickets: recall goes up, precision goes down. F1 goes up too, but 27.7% of the tickets are flagged, more than the senior team can review. F1 cannot see the capacity. The next lesson is about this trade-off.

## 6. Failure case: the most accurate cut-off

Tomás wants "the most accurate checklist". He tries every cut-off from 0 to 5 points and keeps the one with the best accuracy.

> **Predict.** Will the most accurate cut-off flag many tickets or few? Then run the cell.

In [ ]:
for cut_off in range(6):
    report = flag_report(y, points >= cut_off)
    print(cut_off, "| flagged", report["flagged"], "| recall", report["recall"], "| accuracy", report["accuracy"])

> **Check.** The best accuracy is at 4 points or more: `flagged 32 | recall 0.098 | accuracy 0.848`. It catches 18 of the 183 escalations.

Accuracy rewards each correct "no", so the fewer the flags, the higher the accuracy. It does not match Grace's decision: flag at most 20% of tickets and catch as many escalations as possible. **Write down the decision first, then choose the metric from it**: here, precision and recall at a flag rate of at most 20%.

## Next

You can now count the four outcomes of a classifier, compute accuracy, precision, recall and F1, and choose the metric from the decision. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/) has the decision table of real cases and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.